# franka-rs quickstart in simulation

Run these cells from top to bottom with **Shift+Enter**. Each motion cell starts a fresh
headless FR3 simulator, waits until it is ready, connects, then stops it when finished.
You do not need a robot, Desk, or a separate simulator terminal in Binder.

For smaller first steps, open [the Cartesian simulation lab](cartesian_sim_lab.ipynb).
This quickstart explores `move_to`, orientation targets and `follow`. It replays recorded
joint states inline without requiring WebGL; optional Rerun mesh rendering comes last.

Locally, prepare the environment using the repository's `.binder/README.md` first.
These examples deliberately connect only to their own local simulator.

In [ ]:
import os
import time
import numpy as np
import franka
from sim_runtime import simulator
from sim_lab import measured_position, replay

os.environ.setdefault("FRANKA_REALTIME", "ignore")

## Connect and read

The `simulator()` block starts the simulation **before** `Robot` connects. `read_once()`
reads a state and `model()` provides native kinematics. The simulator stops at the end
of the block; the next motion cell gets a fresh starting pose.

In [ ]:
with simulator() as address:
    robot = arm = None
    try:
        robot = franka.Robot(address, realtime="ignore")
        model = robot.model()
        robot.set_collision_behavior_simple([40.] * 7, [40.] * 7, [40.] * 6, [40.] * 6)
        print(robot, "server version", robot.server_version)
        state = robot.read_once()
        print("q (rad)", np.round(state.q, 3))
        print("O_T_EE\n", np.round(state.O_T_EE, 3))
    finally:
        arm = None
        robot = None

## A square with `move_to`

`cartesian_targets` starts the 1 kHz loop on a Rust thread; `move_to` writes a new target
into it whenever this cell gets round to it. Here that is 10 Hz, a 5 cm square in x and y.
`watch` records what happened for the plots below.

In [ ]:
times, qs, targets, measured = [], [], [], []


def watch(arm, seconds):
    """Record time, joint angles, target and measured position at 10 Hz for `seconds`."""
    global segment_t0
    for _ in range(round(seconds * 10)):
        state = arm.state()
        if segment_t0 is None:
            segment_t0 = float(state.time)
        stamp = segment_offset + (float(state.time) - segment_t0)
        # Shared hosts can return the same state twice; record each state once.
        if times and stamp <= times[-1]:
            time.sleep(0.1)
            continue
        times.append(stamp)
        qs.append(state.q)
        targets.append(arm.target()[:3])
        measured.append(measured_position(model, state, "impedance"))
        time.sleep(0.1)


segment_t0 = None
segment_offset = times[-1] + 0.1 if times else 0.0

with simulator() as address:
    robot = arm = None
    try:
        robot = franka.Robot(address, realtime="ignore")
        model = robot.model()
        robot.set_collision_behavior_simple([40.] * 7, [40.] * 7, [40.] * 6, [40.] * 6)
        with robot.cartesian_targets(max_velocity=0.3) as arm:
            start = arm.target()  # position (m) and unit quaternion (x, y, z, w) of the target
            corners = start[:3] + 0.05 * np.array([[0, 0, 0], [1, 0, 0], [1, 1, 0], [0, 1, 0], [0, 0, 0]])
            sides = [np.linspace(a, b, 12, endpoint=False) for a, b in zip(corners, corners[1:])]
            for point in np.concatenate(sides):  # 48 targets, one every 100 ms
                arm.move_to(point)
                watch(arm, 0.1)
    finally:
        arm = None
        robot = None

## A rotation

This cell starts again from the simulator's initial pose. A 7-element target sets
orientation too: position followed by a quaternion `(x, y, z, w)`. Here it adds a
15 degree yaw about the base z axis. `move_by` with 6 elements appends a rotation
vector (axis times angle), here a 10 degree tilt about the base x axis.

In [ ]:
segment_t0 = None
segment_offset = times[-1] + 0.1 if times else 0.0

with simulator() as address:
    robot = arm = None
    try:
        robot = franka.Robot(address, realtime="ignore")
        model = robot.model()
        robot.set_collision_behavior_simple([40.] * 7, [40.] * 7, [40.] * 6, [40.] * 6)
        with robot.cartesian_targets(max_velocity=0.3) as arm:
            pose = arm.target()
            arm.move_to(np.r_[pose[:3], franka.rotated(pose[3:], [0.0, 0.0, np.radians(15.0)])])
            watch(arm, 1.5)
            arm.move_by([0.0, 0.0, 0.0, np.radians(10.0), 0.0, 0.0])
            watch(arm, 1.5)
            print("target orientation", np.round(arm.target()[3:], 3))
    finally:
        arm = None
        robot = None

## Move away, then return with `follow`

In a fresh simulator, first move 3 cm along x. Then `follow` hands a chunk of absolute
targets to a Rust timer thread that publishes one row every `dt`. A policy that predicts
action chunks uses this API. `watch` records each experiment for the replay below.

In [ ]:
segment_t0 = None
segment_offset = times[-1] + 0.1 if times else 0.0

with simulator() as address:
    robot = arm = None
    try:
        robot = franka.Robot(address, realtime="ignore")
        model = robot.model()
        robot.set_collision_behavior_simple([40.] * 7, [40.] * 7, [40.] * 6, [40.] * 6)
        with robot.cartesian_targets(max_velocity=0.3) as arm:
            start = arm.target().copy()
            arm.move_by([0.03, 0.0, 0.0])
            watch(arm, 1.5)
            chunk = np.linspace(arm.target(), start, 20)
            arm.follow(chunk, dt=0.05)
            watch(arm, 2.0)
        print(f"{len(qs)} total samples, final error {np.linalg.norm(measured[-1] - start[:3]) * 1000:.1f} mm")
    finally:
        arm = None
        robot = None

## Replay the motion

Press **Play** or drag the time slider. This view shows the actual recorded simulator
joint states. Each example above resets the robot, so the replay jumps between those
experiments. The inline viewer needs no WebGL or downloaded graphics assets.

In [ ]:
recording = dict(model=model, time=np.array(times) - times[0], q=np.array(qs),
                 measured=np.array(measured), target=np.array(targets))
replay(recording)

The target leads and the measured position follows under the velocity, acceleration and
jerk budget of `cartesian_targets`.

In [ ]:
import matplotlib.pyplot as plt

target, actual = np.array(targets), np.array(measured)
fig, ax = plt.subplots(figsize=(7, 3))
for i, axis in enumerate("xyz"):
    ax.plot(times, target[:, i], f"C{i}", linestyle="--", linewidth=1, label=f"{axis} target")
    ax.plot(times, actual[:, i], f"C{i}", linewidth=1.5, label=f"{axis} measured")
ax.set(xlabel="time (s)", ylabel="position (m)")
ax.legend(ncol=3, fontsize="small", frameon=False)
plt.show()

## Next

`examples/policy_loop.py` is a jittery 6-10 Hz policy loop with a circle, a yaw and tilt
sweep and a `follow` chunk, `examples/rotate.py` the rotation in thirty lines; the
[Python page](https://barisyazici.github.io/franka-rs/getting-started/python.html) of the book has the whole
API, and the [flight recorder](https://barisyazici.github.io/franka-rs/howto/flight-recorder.html)
explains the JSON a `ControlException` leaves behind, which `franka-rerun` replays the same way.

## Optional: Rerun with FR3 meshes

Set `ENABLE_RERUN = True` to use this viewer. The repository includes converted FR3
meshes in `crates/franka-description/meshes/fr3`; the cell finds them automatically
from the notebook directory or repository root. Set `FRANKA_MESHES` to use another
mesh directory. Missing files produce an error with the path and filenames instead
of silently showing a skeleton.

The original meshes are from Franka Robotics' Apache-2.0
[franka_description](https://github.com/frankarobotics/franka_description).
See `tools/franka-meshes/README.md` for conversion and attribution details.
Each mesh uses the frames from `model.link_poses`; the hand uses `model.hand_pose`.

This optional viewer requires WebGL2. The portable playback above remains available
for browsers without it. Use `rr.save("quickstart.rrd")` instead of `notebook_show()`
to write a recording for the desktop viewer.


In [ ]:
ENABLE_RERUN = False

if ENABLE_RERUN:
    from pathlib import Path
    
    import rerun as rr
    
    NAMES = [f"link{k}" for k in range(8)] + ["hand", "finger"]
    if os.environ.get("FRANKA_MESHES"):
        MESHES = Path(os.environ["FRANKA_MESHES"]).expanduser().resolve()
    else:
        MESHES = next(
            (root / "crates/franka-description/meshes/fr3"
             for root in [Path.cwd(), *Path.cwd().parents]
             if (root / "crates/franka-description/meshes/fr3").is_dir()),
            None,
        )
    if MESHES is None:
        raise FileNotFoundError(
            "Bundled FR3 meshes not found. Run from inside the repository, "
            "or set FRANKA_MESHES to a directory containing the FR3 .glb files."
        )
    missing = [f"{name}.glb" for name in NAMES if not (MESHES / f"{name}.glb").is_file()]
    if missing:
        raise FileNotFoundError(f"Missing FR3 mesh files in {MESHES}: {', '.join(missing)}")
    print("Mesh directory:", MESHES)
    FINGER_Z = 0.0584  # the finger frames along the hand's z (franka_description's finger_joint1)
    
    
    def transform(pose):
        return rr.Transform3D(translation=pose[:3, 3], mat3x3=pose[:3, :3])
    
    
    rr.init("franka quickstart")
    rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Z_UP, static=True)
    for name in NAMES[:-1]:
        rr.log(f"world/links/{name}/mesh", rr.Asset3D(path=MESHES / f"{name}.glb"), static=True)
    for finger, yaw in (("finger_left", 0.0), ("finger_right", np.pi)):
        rotation = rr.RotationAxisAngle([0, 0, 1], radians=yaw)
        rr.log(f"world/links/hand/{finger}", rr.Transform3D(translation=[0, 0, FINGER_Z], rotation=rotation), static=True)
        rr.log(f"world/links/hand/{finger}/mesh", rr.Asset3D(path=MESHES / "finger.glb"), static=True)

    for t, q in zip(times, qs):
        rr.set_time("time", duration=t)
        poses = model.link_poses(q)
        for k in range(1, 8):
            rr.log(f"world/links/link{k}", transform(poses[k]))
        rr.log("world/links/hand", transform(model.hand_pose(q)))
    rr.notebook_show(height=500)
